In [ ]:
import os
os.chdir('/mmsegmentation')

# ============================================
# Video → Segmentation+Classification (MMSeg 1.2.2, multitask)
# REALISTIC LOW-LATENCY ADAPTIVE SCHEDULER + BENCHMARK-STYLE GLOBAL MEASUREMENT:
# - realistic sequential pipeline: simulated camera → inference → overlay → display → encode
# - GPU-only measured with CUDA events + synchronize BEFORE and AFTER each sample
# - FIRE/HOLD decided by AdaptiveKeyFrameSelector with a configurable TAU threshold
# - the first frame is always FIRE; then HOLD uses propagated context from the cached keyframe
# - saves mp4v video, NDArray pipeline, and user-friendly OSD
# ============================================

import os, copy, glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from time import perf_counter as now
from tqdm.auto import tqdm
from mmcv.transforms import Compose
from mmseg.utils import register_all_modules
from mmengine.config import Config
from mmengine.registry import init_default_scope
from mmengine.runner.checkpoint import load_checkpoint
from mmseg.registry import MODELS

# ========= PATHS =========
CONFIG = '/mmsegmentation/zmax_configs/04_test_adaptive_scheduler_stateful.py'
CKPT   = '/mmsegmentation/work_dirs/bisenet_adaptive_scheduler_from_pretrain02/best_cls_acc_cls_top1_iter_16200.pth'

DEVICE = 'cuda:0'

# Adjust INPUT_VIDEO if your mp4 is still in the probando_dff folder.
INPUT_VIDEO  = '/0_secuencia_paravideo/1a_secuencia2.mp4'
OUTPUT_VIDEO = '/mmsegmentation/output/video_realistic_adaptive_scheduler_tau003_1a.mp4'
SAVE_VIDEO   = True

# ============= Output / Display =============
TARGET_W, TARGET_H = 1080, 720
DISPLAY_SIMULATE = True
DISPLAY_VSYNC    = True
DISPLAY_FPS      = 10.0  # "real time" at 10 Hz

# ============= Overlay (BGR palette) =============
palette = [
    ('background', [127,127,127]),
    ('red',        [  0,  0,200]),
]
colors = np.array([c for _, c in palette], dtype=np.uint8)
opacity = 0.6
OVERLAY_MODE = 'road_only'

# ============= Classification labels =============
CLS_NAMES = ['LEFT', 'STRAIGHT', 'RIGHT']

# ============= Adaptive scheduler ============
# Main threshold: dev_pred > ADAPTIVE_TAU => FIRE; otherwise, HOLD.
# You can control it from here.
ADAPTIVE_TAU = 0.03

# The first frame is always FIRE.
FIRST_FRAME_FIRE = True

# Force a FIRE if too many consecutive HOLD frames accumulate.
# Leave as None so that only the scheduler makes the decision.
MAX_HOLD_FRAMES = None

# ============= Execution mode =============
BENCHMARK_STYLE_NO_AUTOCAST = True

# ============= Accelerations =============
torch.set_grad_enabled(False)
torch.backends.cudnn.benchmark = False
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

PROCESS_SCALE = 1.0

# trigger_idx now records the temporal decision.
TRIGGER_NAMES = ['HOLD', 'FIRE']

# ============= Metrics (config) =============
METRICS_ENABLE           = True
GPU_SAMPLE_EVERY         = 1
GPU_SAMPLE_OFFSET        = 0
OSD_SHOW_FPS             = True

SAVE_RESULTS = True
RESULTS_DIR  = '/mmsegmentation/output/video_adaptive_scheduler_tau003_1a'
SUMMARY_CSV  = os.path.join(RESULTS_DIR, 'summary_adaptive_scheduler_realistic_global.csv')
DETAILS_CSV  = os.path.join(RESULTS_DIR, 'details_adaptive_scheduler_realistic_per_frame.csv')

# ============= Camera/display simulation ============
PREPROCESS_USE_PINNED    = True
PRED_MASK_DTYPE          = np.uint8
CAMERA_SIMULATE_REALTIME = True
CAMERA_DROP_LATE_FRAMES  = True
CAMERA_SLEEP_WHEN_AHEAD  = False
CAMERA_MAX_CATCHUP_GRABS = 512
DEADLINE_HOLD_ON_LAG     = False


# ------------------------------------------------------------
# Metric utilities
# ------------------------------------------------------------
class Ema:
    def __init__(self, alpha=0.2):
        self.alpha = float(alpha)
        self.value = None
    def update(self, x):
        if x is None:
            return
        self.value = x if self.value is None else (self.alpha * x + (1 - self.alpha) * self.value)


class Profiler:
    """
    Benchmark-style global measurement:
    - GPU-only:
        synchronize -> event_start -> predict -> event_end -> synchronize
      and FPS = 1000 / media_global_ms
    - E2E:
        accumulates per processed frame and is summarized with the global mean
    - It also keeps an overhead/stall calibration reference
      only as auxiliary data.
    """
    def __init__(self,
                 sample_every_gpu=1,
                 sample_offset_gpu=0,
                 use_cuda=True):
        self.sample_every_gpu = max(1, int(sample_every_gpu))
        self.sample_offset_gpu = int(sample_offset_gpu)
        self.use_cuda = bool(use_cuda and torch.cuda.is_available())

        # global accumulators
        self.sum_gpu_ms = 0.0
        self.sum_gpu_adj_ms = 0.0
        self.sum_gpu_stall_ms = 0.0
        self.n_gpu = 0

        self.sum_e2e_ms = 0.0
        self.sum_e2e_disp_ms = 0.0
        self.sum_e2e_enc_ms = 0.0
        self.n_e2e = 0
        self.n_e2e_disp = 0
        self.n_e2e_enc = 0

        # latest value / temporal slope for OSD
        self.last_gpu_ms = None
        self.last_gpu_adj_ms = None
        self.last_gpu_stall_ms = None
        self.last_e2e_ms = None
        self.last_e2e_disp_ms = None
        self.last_e2e_enc_ms = None

        # timing state
        self.ev_start = None
        self.ev_end = None
        self.wall_t0 = None
        self._pending_gpu = None

        # per-frame details
        self.frame_records = []

        # auxiliary calibration
        self.calib_done = False
        self.calib_event_ms = 0.0
        self.calib_stall_ms = 0.0

    def _calibrate(self, iters=30):
        if not self.use_cuda or self.calib_done:
            self.calib_done = True
            return
        torch.cuda.synchronize()
        event_cost = []
        stall_cost = []
        for _ in range(iters):
            e0 = torch.cuda.Event(enable_timing=True)
            e1 = torch.cuda.Event(enable_timing=True)
            t0 = now()
            e0.record()
            e1.record()
            torch.cuda.synchronize()
            t1 = now()
            ms = float(e0.elapsed_time(e1))
            event_cost.append(ms)
            stall_cost.append((t1 - t0) * 1000.0 - ms)
        self.calib_event_ms = float(np.median(event_cost)) if event_cost else 0.0
        self.calib_stall_ms = float(np.median(stall_cost)) if stall_cost else 0.0
        self.calib_done = True

    def want_sample_gpu(self, idx):
        return ((int(idx) + self.sample_offset_gpu) % self.sample_every_gpu) == 0

    def gpu_start(self, idx):
        self._pending_gpu = None
        if not (self.use_cuda and self.want_sample_gpu(idx)):
            return False
        if not self.calib_done:
            self._calibrate()

        # benchmark style: flush the queue before starting the measured section
        torch.cuda.synchronize()
        self.ev_start = torch.cuda.Event(enable_timing=True)
        self.ev_end   = torch.cuda.Event(enable_timing=True)
        self.wall_t0 = now()
        self.ev_start.record()
        return True

    def gpu_end(self):
        if not (self.use_cuda and self.ev_start is not None and self.ev_end is not None and self.wall_t0 is not None):
            self._pending_gpu = None
            return None

        self.ev_end.record()
        torch.cuda.synchronize()
        wall_t1 = now()

        event_ms = float(self.ev_start.elapsed_time(self.ev_end))
        wall_ms = (wall_t1 - self.wall_t0) * 1000.0
        adj_ms = max(1e-6, event_ms - self.calib_event_ms)
        stall_ms = max(0.0, wall_ms - event_ms)

        self.sum_gpu_ms += event_ms
        self.sum_gpu_adj_ms += adj_ms
        self.sum_gpu_stall_ms += stall_ms
        self.n_gpu += 1

        self.last_gpu_ms = event_ms
        self.last_gpu_adj_ms = adj_ms
        self.last_gpu_stall_ms = stall_ms

        self._pending_gpu = {
            'gpu_ms': event_ms,
            'gpu_adj_ms': adj_ms,
            'gpu_stall_ms': stall_ms,
        }

        self.ev_start = None
        self.ev_end = None
        self.wall_t0 = None
        return dict(self._pending_gpu)

    def upd_e2e(self, ms):
        ms = float(ms)
        self.sum_e2e_ms += ms
        self.n_e2e += 1
        self.last_e2e_ms = ms

    def upd_e2e_disp(self, ms):
        ms = float(ms)
        self.sum_e2e_disp_ms += ms
        self.n_e2e_disp += 1
        self.last_e2e_disp_ms = ms

    def upd_e2e_enc(self, ms):
        ms = float(ms)
        self.sum_e2e_enc_ms += ms
        self.n_e2e_enc += 1
        self.last_e2e_enc_ms = ms

    def _mean(self, s, n):
        return (float(s) / float(n)) if n > 0 else None

    def _fps(self, ms):
        return (1000.0 / float(ms)) if (ms is not None and ms > 1e-9) else None

    def mean_gpu(self): return self._mean(self.sum_gpu_ms, self.n_gpu)
    def mean_gpu_adj(self): return self._mean(self.sum_gpu_adj_ms, self.n_gpu)
    def mean_gpu_stall(self): return self._mean(self.sum_gpu_stall_ms, self.n_gpu)
    def mean_e2e(self): return self._mean(self.sum_e2e_ms, self.n_e2e)
    def mean_e2e_disp(self): return self._mean(self.sum_e2e_disp_ms, self.n_e2e_disp)
    def mean_e2e_enc(self): return self._mean(self.sum_e2e_enc_ms, self.n_e2e_enc)

    def fps_gpu(self): return self._fps(self.mean_gpu())
    def fps_gpu_adj(self): return self._fps(self.mean_gpu_adj())
    def fps_e2e(self): return self._fps(self.mean_e2e())
    def fps_e2e_disp(self): return self._fps(self.mean_e2e_disp())
    def fps_e2e_enc(self): return self._fps(self.mean_e2e_enc())

    def commit_frame(self, **kwargs):
        rec = dict(kwargs)
        gpu = self._pending_gpu if self._pending_gpu is not None else {}
        rec['gpu_ms'] = float(gpu['gpu_ms']) if 'gpu_ms' in gpu else np.nan
        rec['gpu_adj_ms'] = float(gpu['gpu_adj_ms']) if 'gpu_adj_ms' in gpu else np.nan
        rec['gpu_stall_ms'] = float(gpu['gpu_stall_ms']) if 'gpu_stall_ms' in gpu else np.nan
        self.frame_records.append(rec)
        self._pending_gpu = None

    def details_df(self):
        if len(self.frame_records) == 0:
            return pd.DataFrame(columns=[
                'frame_idx', 'phase', 'cls_idx', 'trigger_idx', 'trigger_name',
                'mode_changed', 'active_k_before', 'active_k_after',
                'next_fire_idx_after', 'dropped_src_total',
                'e2e_infer_ms', 'e2e_infer_display_ms', 'e2e_infer_display_encode_ms',
                'gpu_ms', 'gpu_adj_ms', 'gpu_stall_ms',
            ])
        return pd.DataFrame(self.frame_records)

    def summary_df(self, *,
                   method_name,
                   input_video,
                   output_video,
                   fps_src,
                   w_src, h_src,
                   target_w, target_h,
                   processed, written, dropped_src,
                   fire_cnt, hold_cnt, errors,
                   last_src_idx,
                   k_straight, k_curve,
                   final_trigger_name,
                   next_fire_idx_final,
                   config_path,
                   ckpt_path):
        rows = []

        def _append_row(metric_name, mean_ms, n, values=None, extra=None):
            if mean_ms is None or n <= 0:
                return
            row = {
                'method': method_name,
                'metric': metric_name,
                'n_frames': int(n),
                'mean_ms': float(mean_ms),
                'fps_from_mean': float(1000.0 / mean_ms),
                'processed_frames': int(processed),
                'written_frames': int(written),
                'dropped_src_frames': int(dropped_src),
                'fires': int(fire_cnt),
                'holds': int(hold_cnt),
                'fire_ratio': float(fire_cnt / max(1, processed)),
                'hold_ratio': float(hold_cnt / max(1, processed)),
                'errors': int(errors),
                'source_fps': float(fps_src),
                'last_src_idx': int(last_src_idx),
                'input_w': int(w_src),
                'input_h': int(h_src),
                'output_w': int(target_w),
                'output_h': int(target_h),
                'k_straight': int(k_straight),
                'k_curve': int(k_curve),
                'final_trigger': str(final_trigger_name),
                'next_fire_idx_final': int(next_fire_idx_final),
                'video_path': input_video,
                'output_video_path': output_video,
                'config_path': config_path,
                'ckpt_path': ckpt_path,
            }
            if values is not None and len(values) > 0:
                arr = np.asarray(values, dtype=np.float64)
                row.update({
                    'median_ms': float(np.median(arr)),
                    'p95_ms': float(np.percentile(arr, 95)),
                    'min_ms': float(arr.min()),
                    'max_ms': float(arr.max()),
                })
            if extra:
                row.update(extra)
            rows.append(row)

        df = self.details_df()

        if len(df) > 0:
            gpu_vals = df['gpu_ms'].dropna().astype(float).to_numpy() if 'gpu_ms' in df else np.array([])
            gpu_adj_vals = df['gpu_adj_ms'].dropna().astype(float).to_numpy() if 'gpu_adj_ms' in df else np.array([])
            e2e_vals = df['e2e_infer_ms'].dropna().astype(float).to_numpy() if 'e2e_infer_ms' in df else np.array([])
            e2e_disp_vals = df['e2e_infer_display_ms'].dropna().astype(float).to_numpy() if 'e2e_infer_display_ms' in df else np.array([])
            e2e_enc_vals = df['e2e_infer_display_encode_ms'].dropna().astype(float).to_numpy() if 'e2e_infer_display_encode_ms' in df else np.array([])

            _append_row(
                'GPU_ONLY_GLOBAL',
                float(gpu_vals.mean()) if gpu_vals.size else None,
                int(gpu_vals.size),
                values=gpu_vals,
                extra={
                    'calib_event_overhead_ms': float(self.calib_event_ms),
                    'calib_stall_ref_ms': float(self.calib_stall_ms),
                    'runtime_mean_stall_ms': float(self.mean_gpu_stall()) if self.mean_gpu_stall() is not None else np.nan,
                }
            )
            _append_row(
                'GPU_ONLY_ADJ_GLOBAL',
                float(gpu_adj_vals.mean()) if gpu_adj_vals.size else None,
                int(gpu_adj_vals.size),
                values=gpu_adj_vals,
                extra={
                    'calib_event_overhead_ms': float(self.calib_event_ms),
                    'calib_stall_ref_ms': float(self.calib_stall_ms),
                    'runtime_mean_stall_ms': float(self.mean_gpu_stall()) if self.mean_gpu_stall() is not None else np.nan,
                }
            )
            _append_row('E2E_INFER_GLOBAL', float(e2e_vals.mean()) if e2e_vals.size else None, int(e2e_vals.size), values=e2e_vals)
            _append_row('E2E_INFER_DISPLAY_GLOBAL', float(e2e_disp_vals.mean()) if e2e_disp_vals.size else None, int(e2e_disp_vals.size), values=e2e_disp_vals)
            _append_row('E2E_INFER_DISPLAY_ENCODE_GLOBAL', float(e2e_enc_vals.mean()) if e2e_enc_vals.size else None, int(e2e_enc_vals.size), values=e2e_enc_vals)

        return pd.DataFrame(rows)


PROF = Profiler(sample_every_gpu=GPU_SAMPLE_EVERY,
                sample_offset_gpu=GPU_SAMPLE_OFFSET,
                use_cuda=True)

# ------------------------------------------------------------
# 1) NDArray pipeline without annotations
# ------------------------------------------------------------

# ------------------------------------------------------------
# 2) Multitask NDArray inference (OPTIMIZED)
#   - without Compose / PackSegInputs
#   - realistic manual preprocessing for online camera/video
#   - uses model.predict(...) with minimal data_samples
# ------------------------------------------------------------
try:
    from mmseg.structures.dual_task_seg_data_sample import DualTaskSegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample

def _label_from_LabelData(lbl):
    import numpy as _np
    if lbl is None:
        return None, None

    x = lbl
    scores = None
    for _ in range(8):
        if x is None:
            return None, scores

        if torch.is_tensor(x):
            return int(x.reshape(-1)[0].detach().cpu().item()), scores

        if isinstance(x, _np.ndarray):
            return int(x.reshape(-1)[0]), scores

        if isinstance(x, (int, float, bool, _np.integer, _np.floating)):
            return int(x), scores

        if isinstance(x, (list, tuple)):
            if len(x) == 0:
                return None, scores
            x = x[0]
            continue

        if isinstance(x, dict):
            if 'score' in x and x['score'] is not None:
                s = x['score']
                if torch.is_tensor(s):
                    scores = s.detach().cpu().numpy()
                else:
                    scores = _np.asarray(s)
                if scores.size > 0:
                    return int(scores.reshape(-1).argmax()), scores
            for k in ('label', 'pred_label', 'data', 'value'):
                if k in x:
                    x = x[k]
                    break
            else:
                return None, scores
            continue

        if hasattr(x, 'score') and getattr(x, 'score') is not None:
            s = getattr(x, 'score')
            if torch.is_tensor(s):
                scores = s.detach().cpu().numpy()
            else:
                scores = _np.asarray(s)
            if scores.size > 0:
                return int(scores.reshape(-1).argmax()), scores

        moved = False
        for attr in ('label', 'data', 'value'):
            if hasattr(x, attr):
                x = getattr(x, attr)
                moved = True
                break
        if moved:
            continue

        if hasattr(x, 'item') and callable(x.item):
            try:
                return int(x.item()), scores
            except Exception:
                pass

        break

    return None, scores

def _safe_int_from_any(x):
    y, _ = _label_from_LabelData(x)
    return y

def _trigger_from_sample(sample):
    # pred_trigger_label has priority
    if hasattr(sample, 'pred_trigger_label'):
        idx, scores = _label_from_LabelData(sample.pred_trigger_label)
        if idx is not None:
            return idx, scores

    # pred_trigger as fallback
    if hasattr(sample, 'pred_trigger'):
        idx, scores = _label_from_LabelData(sample.pred_trigger)
        if idx is not None:
            return idx, scores

    return None, None



class AdaptiveSchedulerStatefulInferencer:
    """Stateful inference for BiSeNetAdaptiveVideoSegmentor.

    FIRE:
        runs Spatial Path + Context Path + FFM, updates the cache.
    HOLD:
        runs the current Spatial Path + feature_propagation(context_key, spatial_key, spatial_cur) + FFM.

    The decision is made with:
        dev_pred = keyframe_selector(S_key, S_t)
        FIRE if dev_pred > tau.
    """

    def __init__(self, model, tau=0.03, max_hold_frames=None, first_frame_fire=True):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        self.tau = float(tau)
        self.max_hold_frames = None if max_hold_frames is None else int(max_hold_frames)
        self.first_frame_fire = bool(first_frame_fire)

        self.cache = None
        self.frame_counter = 0
        self.hold_count_since_fire = 0
        self.last_phase = 'INIT'
        self.last_dev_pred = np.nan
        self.last_fire_reason = 'INIT'

        # Input size from the stateful config, if available.
        self.input_w = 512
        self.input_h = 512
        try:
            st_cfg = model.cfg.get('adaptive_stateful_cfg', {})
            size = st_cfg.get('input_size', (512, 512))
            self.input_w = int(size[0])
            self.input_h = int(size[1])
        except Exception:
            pass

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std  = np.array(cfg_dp.get('std',  [58.395, 57.12, 57.375]), dtype=np.float32)
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std  = torch.tensor(std,  device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, device=self.device)
            self._gpu_chw_f32 = torch.empty((1, 3, self.input_h, self.input_w), dtype=torch.float32, device=self.device)

    def reset_state(self):
        self.cache = None
        self.frame_counter = 0
        self.hold_count_since_fire = 0
        self.last_phase = 'INIT'
        self.last_dev_pred = np.nan
        self.last_fire_reason = 'RESET'

    def set_tau(self, tau):
        self.tau = float(tau)

    def _preprocess(self, img_bgr_nd):
        if (img_bgr_nd.shape[1], img_bgr_nd.shape[0]) != (self.input_w, self.input_h):
            cv2.resize(img_bgr_nd, (self.input_w, self.input_h), dst=self._resize_hwc, interpolation=cv2.INTER_LINEAR)
            img = self._resize_hwc
        else:
            img = img_bgr_nd

        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img)
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            chw_u8 = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            self._gpu_chw_f32.copy_(chw_u8)
            self._gpu_chw_f32.sub_(self.mean).div_(self.std)
            return self._gpu_chw_f32

        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).unsqueeze(0)
        if self.use_cuda:
            tensor = tensor.to(self.device, non_blocking=True)
        tensor = tensor.float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    def _decode_from_fuse(self, x_fuse, context8_like, context16_like, spatial_like):
        feats = (x_fuse, context8_like, context16_like, spatial_like)
        seg_logits = self.model.decode_head.forward(feats)
        if seg_logits.shape[-2:] != (self.input_h, self.input_w):
            seg_logits = F.interpolate(
                seg_logits,
                size=(self.input_h, self.input_w),
                mode='bilinear',
                align_corners=False
            )
        pred_mask = seg_logits.argmax(dim=1)[0].detach().to(torch.uint8).cpu().numpy()

        cls_idx = None
        if self.model.cls_head is not None:
            cls_logits = self.model.cls_head.forward(x_fuse)
            cls_idx = int(torch.argmax(cls_logits, dim=1)[0].detach().cpu().item())

        return pred_mask, cls_idx

    def _run_full_fire(self, inputs, spatial_cur=None):
        # If spatial_cur was already computed for the scheduler, reuse it.
        x_context8, x_context16 = self.model.backbone.context_path(inputs)
        if spatial_cur is None:
            spatial_cur = self.model.backbone.spatial_path(inputs)
        x_fuse = self.model.backbone.ffm(spatial_cur, x_context8)

        self.cache = dict(
            spatial_key=spatial_cur.detach(),
            context8_key=x_context8.detach(),
            context16_key=x_context16.detach(),
        )

        pred_mask, cls_idx = self._decode_from_fuse(x_fuse, x_context8, x_context16, spatial_cur)
        return pred_mask, cls_idx

    def _run_hold_prop(self, spatial_cur):
        context8_prop = self.model.feature_propagation(
            context_key=self.cache['context8_key'],
            spatial_key=self.cache['spatial_key'],
            spatial_cur=spatial_cur
        )
        context16_key = self.cache.get('context16_key', context8_prop)
        x_fuse = self.model.backbone.ffm(spatial_cur, context8_prop)
        pred_mask, cls_idx = self._decode_from_fuse(x_fuse, context8_prop, context16_key, spatial_cur)
        return pred_mask, cls_idx

    @torch.inference_mode()
    def __call__(self, img_bgr_nd, idx=None, profiler: Profiler=None):
        inputs = self._preprocess(img_bgr_nd)

        sampled = False
        if profiler is not None:
            sampled = profiler.gpu_start(idx)

        dev_pred = np.nan
        do_fire = False
        fire_reason = ''

        # First frame or missing cache: mandatory FIRE.
        if self.cache is None:
            do_fire = True
            fire_reason = 'FIRST_OR_EMPTY_CACHE'
            spatial_cur = None
        else:
            spatial_cur = self.model.backbone.spatial_path(inputs)

            if self.model.keyframe_selector is None:
                dev_tensor = torch.tensor([1.0], device=self.device)
            else:
                dev_tensor = self.model.keyframe_selector(self.cache['spatial_key'], spatial_cur)

            dev_pred = float(dev_tensor.detach().cpu().flatten()[0].item())

            if dev_pred > self.tau:
                do_fire = True
                fire_reason = 'DEV_GT_TAU'
            elif self.max_hold_frames is not None and self.hold_count_since_fire >= self.max_hold_frames:
                do_fire = True
                fire_reason = 'MAX_HOLD'
            else:
                do_fire = False
                fire_reason = 'DEV_LE_TAU'

        if do_fire:
            pred_mask, cls_idx = self._run_full_fire(inputs, spatial_cur=spatial_cur if 'spatial_cur' in locals() else None)
            self.hold_count_since_fire = 0
            self.last_phase = 'FIRE'
        else:
            pred_mask, cls_idx = self._run_hold_prop(spatial_cur)
            self.hold_count_since_fire += 1
            self.last_phase = 'HOLD'

        self.last_dev_pred = dev_pred
        self.last_fire_reason = fire_reason
        self.frame_counter += 1

        if sampled and profiler is not None:
            profiler.gpu_end()

        trigger_idx = 1 if self.last_phase == 'FIRE' else 0
        return pred_mask, cls_idx, trigger_idx


# ------------------------------------------------------------
# 3) Internal adaptive scheduler by TAU
# ------------------------------------------------------------
# ------------------------------------------------------------
# 3) FIXED DFF (external scheduler + internal FIRE/HOLD cache)
# (deep feature cache from the CONTEXT PATH)
# ------------------------------------------------------------
class _UnusedContextPathClock:
    """
    Patches BiSeNetV1.context_path.forward(x):
      - FIRE: runs and caches (x_context8, x_context16)
      - HOLD: returns the cached tuple (without running the context backbone/ARM/heads)
    The rest (spatial_path + ffm) and heads ALWAYS run ⇒ new mask for each frame.
    """
    def __init__(self, bise: torch.nn.Module):
        assert hasattr(bise, 'context_path') and hasattr(bise, 'spatial_path'), \
            "El backbone no parece ser BiSeNetV1 con context_path/spatial_path."
        self.context_path = bise.context_path
        self.cache = None
        self.hold = False
        self._orig_forward = self.context_path.forward
        self._install()

    def _install(self):
        @torch.inference_mode()
        def wrapped_forward(x):
            if self.hold and (self.cache is not None):
                return self.cache
            out = self._orig_forward(x)  # tuple: (x_16_up, x_32_up)
            if isinstance(out, (list, tuple)):
                self.cache = tuple(o.detach() if torch.is_tensor(o) else o for o in out)
            elif torch.is_tensor(out):
                self.cache = (out.detach(),)
            else:
                self.cache = out
            return out
        self.context_path.forward = wrapped_forward

    def set_hold(self, flag: bool):
        self.hold = bool(flag)

    def invalidate(self):
        self.cache = None

# ------------------------------------------------------------
# 3b) Fixed DFF scheduler
# ------------------------------------------------------------
class _UnusedAdaptiveScheduler:
    def __init__(self, k_straight=100, k_curve=30, default_mode='curve'):
        self.k_straight = int(k_straight)
        self.k_curve = int(k_curve)
        self.active_k = self.k_curve if str(default_mode).lower() == 'curve' else self.k_straight
        self.next_fire_idx = 0
        self.last_trigger_idx = None

    def should_fire(self, frame_idx: int) -> bool:
        return int(frame_idx) >= int(self.next_fire_idx)

    def _k_from_trigger(self, trigger_idx):
        if trigger_idx == TRIGGER_IDX_CURVE:
            return self.k_curve
        if trigger_idx == TRIGGER_IDX_STRAIGHT:
            return self.k_straight
        return self.active_k

    def update_after_inference(self, frame_idx: int, trigger_idx, did_fire: bool):
        desired_k = self._k_from_trigger(trigger_idx)
        mode_changed = (desired_k != self.active_k)

        self.last_trigger_idx = trigger_idx
        self.active_k = desired_k

        # Policy equivalent to the benchmark:
        # if the mode changes (STRAIGHT <-> CURVE), the next frame is forced as FIRE.
        if mode_changed:
            self.next_fire_idx = int(frame_idx) + 1
            return

        if did_fire:
            self.next_fire_idx = int(frame_idx) + int(self.active_k)

    def trigger_name(self):
        if self.last_trigger_idx is None:
            return 'UNKNOWN'
        if 0 <= int(self.last_trigger_idx) < len(TRIGGER_NAMES):
            return TRIGGER_NAMES[int(self.last_trigger_idx)]
        return str(self.last_trigger_idx)


class DFFFixedScheduler:
    def __init__(self, k=100):
        self.k = int(k)
        self.active_k = self.k
        self.next_fire_idx = 0
        self.last_trigger_idx = None

    def should_fire(self, frame_idx: int) -> bool:
        return int(frame_idx) >= int(self.next_fire_idx)

    def update_after_inference(self, frame_idx: int, trigger_idx, did_fire: bool):
        self.last_trigger_idx = trigger_idx
        self.active_k = self.k
        if did_fire:
            self.next_fire_idx = int(frame_idx) + self.k

    def trigger_name(self):
        if self.last_trigger_idx is None:
            return 'UNKNOWN'
        if 0 <= int(self.last_trigger_idx) < len(TRIGGER_NAMES):
            return TRIGGER_NAMES[int(self.last_trigger_idx)]
        return str(self.last_trigger_idx)

# ------------------------------------------------------------
# 4) Simulated display (optional vsync rhythm)
# ------------------------------------------------------------
class DisplaySimulator:
    def __init__(self, width, height, vsync=True, fps=60.0):
        self.tw, self.th = int(width), int(height)
        self.vsync = bool(vsync)
        self.period = (1.0 / float(fps)) if fps and fps > 0 else 0.0
        self.clock0 = now()
        self.index = 0
        self.fb = np.empty((self.th, self.tw, 3), dtype=np.uint8)
        self.resize_buf = np.empty((self.th, self.tw, 3), dtype=np.uint8)

    def present(self, frame_bgr):
        if (frame_bgr.shape[1], frame_bgr.shape[0]) != (self.tw, self.th):
            cv2.resize(
                frame_bgr,
                (self.tw, self.th),
                dst=self.resize_buf,
                interpolation=cv2.INTER_LINEAR
            )
            src = self.resize_buf
        else:
            src = frame_bgr

        # same semantics as an "RGB framebuffer", but without temporary allocations
        cv2.cvtColor(src, cv2.COLOR_BGR2RGB, dst=self.fb)

        if self.vsync and self.period > 0:
            next_deadline = self.clock0 + (self.index + 1) * self.period
            t = now()
            if t < next_deadline:
                import time as _t
                _t.sleep(max(0.0, next_deadline - t))
            self.index += 1

        return src  # BGR at TARGET resolution


# ------------------------------------------------------------
# 5) Centered OSD text (OPTIMIZED)
#   - caches scale and sizes per text
# ------------------------------------------------------------
class CenterTopDirectionPainter:
    def __init__(self, icon_w=110, icon_h=95, color=(0,0,255), thickness=10):
        self.icon_w = int(icon_w)
        self.icon_h = int(icon_h)
        self.color = tuple(int(v) for v in color)
        self.thickness = int(thickness)

    def _dark_box(self, img_bgr, x0, y0, x1, y1):
        roi = img_bgr[y0:y1, x0:x1]
        if roi.size > 0:
            overlay = roi.copy()
            cv2.rectangle(overlay, (0, 0), (roi.shape[1]-1, roi.shape[0]-1), (0,0,0), -1)
            roi[:] = cv2.addWeighted(overlay, 0.35, roi, 0.65, 0)

    def _draw_straight(self, img_bgr, x0, y0, x1, y1):
        w = x1 - x0
        h = y1 - y0
        xc = x0 + w // 2
        yb = y0 + int(0.88 * h)
        yt = y0 + int(0.18 * h)
        cv2.arrowedLine(
            img_bgr,
            (xc, yb),
            (xc, yt),
            self.color,
            self.thickness,
            cv2.LINE_AA,
            tipLength=0.28
        )

    def _draw_left(self, img_bgr, x0, y0, x1, y1):
        w = x1 - x0
        h = y1 - y0
        xc = x0 + int(0.62 * w)
        yb = y0 + int(0.88 * h)
        yt = y0 + int(0.22 * h)
        xl = x0 + int(0.18 * w)
        cv2.line(img_bgr, (xc, yb), (xc, yt), self.color, self.thickness, cv2.LINE_AA)
        cv2.arrowedLine(
            img_bgr,
            (xc, yt),
            (xl, yt),
            self.color,
            self.thickness,
            cv2.LINE_AA,
            tipLength=0.30
        )

    def _draw_right(self, img_bgr, x0, y0, x1, y1):
        w = x1 - x0
        h = y1 - y0
        xc = x0 + int(0.38 * w)
        yb = y0 + int(0.88 * h)
        yt = y0 + int(0.22 * h)
        xr = x0 + int(0.82 * w)
        cv2.line(img_bgr, (xc, yb), (xc, yt), self.color, self.thickness, cv2.LINE_AA)
        cv2.arrowedLine(
            img_bgr,
            (xc, yt),
            (xr, yt),
            self.color,
            self.thickness,
            cv2.LINE_AA,
            tipLength=0.30
        )

    def draw(self, img_bgr, direction, color=None):
        H, W = img_bgr.shape[:2]
        icon_w = min(self.icon_w, max(60, W - 20))
        icon_h = min(self.icon_h, max(50, H - 20))
        x0 = max(8, int((W - icon_w) / 2))
        y0 = 8
        x1 = min(W - 8, x0 + icon_w)
        y1 = min(H - 8, y0 + icon_h)

        self._dark_box(img_bgr, x0, y0, x1, y1)

        label = str(direction).strip().upper()
        if label == 'LEFT':
            self._draw_left(img_bgr, x0, y0, x1, y1)
        elif label == 'RIGHT':
            self._draw_right(img_bgr, x0, y0, x1, y1)
        else:
            self._draw_straight(img_bgr, x0, y0, x1, y1)

        return img_bgr

_LABEL_PAINTER = CenterTopDirectionPainter(icon_w=600, icon_h=400, color=(255,255,0), thickness=80)
_ROAD_COLOR = colors[1].astype(np.uint8)

class FastOverlayRenderer:
    def __init__(self, road_color, alpha, mode='road_only'):
        self.road_color = np.asarray(road_color, dtype=np.uint8)
        self.alpha = float(alpha)
        self.mode = str(mode)
        self.alpha_u16 = int(round(self.alpha * 256.0))
        self.inv_alpha_u16 = 256 - self.alpha_u16
        self.out = None
        self.road = None
        self.size = None

    def _ensure(self, h, w):
        if self.size != (h, w):
            self.size = (h, w)
            self.out = np.empty((h, w, 3), dtype=np.uint8)
            self.road = np.empty((h, w), dtype=np.uint8)

    def road_only(self, img_bgr, pred_small):
        h, w = img_bgr.shape[:2]
        self._ensure(h, w)
        np.copyto(self.out, img_bgr)

        if pred_small.dtype == np.uint8:
            road_small = pred_small
        else:
            road_small = (pred_small == 1).astype(np.uint8)

        if (road_small.shape[1] != w) or (road_small.shape[0] != h):
            cv2.resize(
                road_small,
                (w, h),
                dst=self.road,
                interpolation=cv2.INTER_NEAREST
            )
        else:
            np.copyto(self.road, road_small)

        mask = self.road != 0
        if not np.any(mask):
            return self.out

        # in-place integer blending to avoid large temporary float32 arrays
        for c, color_c in enumerate(self.road_color):
            chan = self.out[..., c]
            vals = chan[mask].astype(np.uint16)
            chan[mask] = ((vals * self.inv_alpha_u16 + int(color_c) * self.alpha_u16 + 128) >> 8).astype(np.uint8)

        return self.out

    def full_palette(self, img_bgr, pred_small):
        h, w = img_bgr.shape[:2]
        mask_small = colors[pred_small]
        mask = (cv2.resize(mask_small, (w, h), interpolation=cv2.INTER_NEAREST)
                if (mask_small.shape[1] != w or mask_small.shape[0] != h) else mask_small)
        return cv2.addWeighted(img_bgr, opacity, mask, 1 - opacity, 0)

    def __call__(self, img_bgr, pred_small):
        if self.mode == 'road_only':
            return self.road_only(img_bgr, pred_small)
        return self.full_palette(img_bgr, pred_small)

_OVERLAY = FastOverlayRenderer(_ROAD_COLOR, opacity, mode=OVERLAY_MODE)

def apply_seg_overlay(img_bgr, pred_small):
    return _OVERLAY(img_bgr, pred_small)

# ------------------------------------------------------------
# 6) Frame processing (OPTIMIZED)
# ------------------------------------------------------------
@torch.inference_mode()
def process_frame(img_bgr, idx=None, profiler: Profiler=None, infer_call=None):
    small = img_bgr if PROCESS_SCALE == 1.0 else cv2.resize(
        img_bgr, None, fx=PROCESS_SCALE, fy=PROCESS_SCALE, interpolation=cv2.INTER_AREA)

    use_autocast = torch.cuda.is_available() and (not BENCHMARK_STYLE_NO_AUTOCAST)

    if use_autocast:
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            pred_small, cls_idx, trigger_idx = infer_call(small, idx=idx, profiler=profiler)
    else:
        pred_small, cls_idx, trigger_idx = infer_call(small, idx=idx, profiler=profiler)

    out = apply_seg_overlay(img_bgr, pred_small)

    label = CLS_NAMES[cls_idx] if (cls_idx is not None and 0 <= cls_idx < len(CLS_NAMES)) else "UNKNOWN"
    out = _LABEL_PAINTER.draw(out, label)

    return out, cls_idx, trigger_idx

# ------------------------------------------------------------
# 7) Warm-up (model) – limited
# ------------------------------------------------------------
def warmup_full_loop(input_video_path, max_frames=60, infer_call=None):
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir video para warm-up: {input_video_path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    pbar = tqdm(total=min(total, max_frames) if total>0 else max_frames,
                unit="frame", desc="Warm-up (modelo DFF bench)")
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok or i >= max_frames:
            break
        _, _, _ = process_frame(frame, idx=i, profiler=None, infer_call=infer_call)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        pbar.update(1); i += 1
    pbar.close()
    cap.release()

# ------------------------------------------------------------
# 7b) Video source with real camera behavior
#   - single-threaded, without prefetching or queues
#   - if running late, jump to recent frames (like a camera buffer)
# ------------------------------------------------------------
class LiveVideoSource:
    def __init__(self, cap, fps, realtime=True, drop_late_frames=True,
                 sleep_when_ahead=False, max_catchup_grabs=512):
        self.cap = cap
        self.fps = float(fps)
        self.period = 1.0 / self.fps if self.fps > 0 else 0.0
        self.realtime = bool(realtime and self.period > 0)
        self.drop_late_frames = bool(drop_late_frames)
        self.sleep_when_ahead = bool(sleep_when_ahead)
        self.max_catchup_grabs = max(1, int(max_catchup_grabs))
        self.clock0 = now()
        self.next_src_idx = 0

    def _target_src_idx(self):
        if not self.realtime:
            return self.next_src_idx
        elapsed = max(0.0, now() - self.clock0)
        return int(elapsed / self.period)

    def read(self):
        if self.realtime and self.sleep_when_ahead:
            target = self._target_src_idx()
            if self.next_src_idx > target:
                import time as _t
                next_deadline = self.clock0 + self.next_src_idx * self.period
                t = now()
                if t < next_deadline:
                    _t.sleep(max(0.0, next_deadline - t))

        dropped = 0
        if self.realtime and self.drop_late_frames:
            target = self._target_src_idx()
            late = max(0, target - self.next_src_idx)
            if late > 0:
                late = min(late, self.max_catchup_grabs)
                for _ in range(late):
                    if not self.cap.grab():
                        return False, None, self.next_src_idx, dropped
                    self.next_src_idx += 1
                    dropped += 1

        ok, frame = self.cap.read()
        if not ok:
            return False, None, self.next_src_idx, dropped

        src_idx = self.next_src_idx
        self.next_src_idx += 1
        return True, frame, src_idx, dropped


# ------------------------------------------------------------
# 8) Main stateful adaptive loop + E2E on ALL frames + sampled GPU-only
# ------------------------------------------------------------
def ensure_fps(x):
    try:
        x = float(x)
    except:
        return 25.0
    return 25.0 if x <= 1.0 else x

def video_to_video_preserve_duration(input_path, output_path=None,
                                     enforce_deadline=True,
                                     write_unprocessed_on_error=True,
                                     annotate_osd=False,
                                     fps_override=None,
                                     infer_call=None,
                                     ctx_clock=None,
                                     scheduler=None):
    assert callable(infer_call), "Debes pasar infer_call (AdaptiveSchedulerStatefulInferencer.__call__)."

    filehead = os.path.basename(input_path)
    output_path = output_path or f'out-keepdur-{filehead}'

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f'No se pudo abrir: {input_path}')

    fps_src = ensure_fps(fps_override if fps_override is not None else cap.get(cv2.CAP_PROP_FPS))
    period = 1.0 / fps_src
    w_src = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_src = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count_prop = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0

    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_src, (TARGET_W, TARGET_H)) if SAVE_VIDEO else None
    if SAVE_VIDEO and not out.isOpened():
        raise RuntimeError(f'No se pudo abrir VideoWriter para: {output_path}')

    disp = DisplaySimulator(
        TARGET_W, TARGET_H,
        vsync=DISPLAY_VSYNC,
        fps=DISPLAY_FPS
    ) if DISPLAY_SIMULATE else None

    source = LiveVideoSource(
        cap, fps_src,
        realtime=CAMERA_SIMULATE_REALTIME,
        drop_late_frames=CAMERA_DROP_LATE_FRAMES,
        sleep_when_ahead=(CAMERA_SLEEP_WHEN_AHEAD and disp is None),
        max_catchup_grabs=CAMERA_MAX_CATCHUP_GRABS
    )

    fire_cnt = 0
    hold_cnt = 0
    errors = 0
    written = 0
    dropped_src = 0
    processed = 0
    last_src_idx = -1
    previous_phase = None

    t0_pipeline = now()

    pbar = tqdm(total=frame_count_prop if frame_count_prop > 0 else None,
                desc=f'Video→Seg+Cls (Adaptive scheduler tau={ADAPTIVE_TAU:.4f}, mp4v)')

    try:
        while True:
            success, frame, src_frame_idx, dropped_now = source.read()
            if not success:
                break

            dropped_src += int(dropped_now)
            last_src_idx = int(src_frame_idx)

            t_e2e0 = now()

            try:
                frame_proc, cls_idx, trigger_idx = process_frame(
                    frame, idx=src_frame_idx,
                    profiler=PROF if METRICS_ENABLE else None,
                    infer_call=infer_call
                )

                phase = str(getattr(infer_call, 'last_phase', 'UNKNOWN'))
                dev_pred = float(getattr(infer_call, 'last_dev_pred', np.nan))
                fire_reason = str(getattr(infer_call, 'last_fire_reason', 'UNKNOWN'))
                tau = float(getattr(infer_call, 'tau', ADAPTIVE_TAU))
                hold_age = int(getattr(infer_call, 'hold_count_since_fire', 0))

                do_fire = (phase == 'FIRE')
                if do_fire:
                    fire_cnt += 1
                else:
                    hold_cnt += 1

                mode_changed = (previous_phase is not None and phase != previous_phase)
                previous_phase = phase

            except Exception as e:
                errors += 1
                print('error!', e)
                frame_proc = frame if write_unprocessed_on_error else None
                if frame_proc is None:
                    processed += 1
                    pbar.update(int(dropped_now) + 1)
                    continue
                cls_idx = None
                trigger_idx = None
                phase = 'ERROR'
                do_fire = False
                dev_pred = np.nan
                fire_reason = 'ERROR'
                tau = float(ADAPTIVE_TAU)
                hold_age = -1
                mode_changed = False

            t_infer1 = now()
            e2e_infer_ms = (t_infer1 - t_e2e0) * 1000.0
            if METRICS_ENABLE:
                PROF.upd_e2e(e2e_infer_ms)

            if disp is not None:
                frame_for_write = disp.present(frame_proc)
            else:
                if (frame_proc.shape[1], frame_proc.shape[0]) != (TARGET_W, TARGET_H):
                    frame_for_write = cv2.resize(frame_proc, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
                else:
                    frame_for_write = frame_proc

            t_disp1 = now()
            e2e_disp_ms = (t_disp1 - t_e2e0) * 1000.0
            if METRICS_ENABLE:
                PROF.upd_e2e_disp(e2e_disp_ms)

            if annotate_osd:
                elapsed = max(1e-6, now() - t0_pipeline)
                fps_eff = (written / elapsed) if elapsed > 0 else 0.0
                txts = [f"{phase} | dev:{dev_pred:.4f} tau:{tau:.4f} | reason:{fire_reason} | src:{src_frame_idx} | drop:{dropped_src} | fire:{fire_cnt} hold:{hold_cnt} err:{errors} fps_eff:{fps_eff:.1f}"]
                if OSD_SHOW_FPS and METRICS_ENABLE:
                    fgpu = PROF.fps_gpu(); fgpuA = PROF.fps_gpu_adj()
                    fe2e = PROF.fps_e2e(); fdis = PROF.fps_e2e_disp(); fenc = PROF.fps_e2e_enc()
                    small = []
                    if fgpu:  small.append(f"GPU:{fgpu:.1f}")
                    if fgpuA: small.append(f"GPUadj:{fgpuA:.1f}")
                    if fe2e:  small.append(f"E2E:{fe2e:.1f}")
                    if fdis:  small.append(f"DSP:{fdis:.1f}")
                    if fenc:  small.append(f"ENC:{fenc:.1f}")
                    if small:
                        txts.append(" | ".join(small) + " | AVG-global")
                cv2.putText(frame_for_write, "  ||  ".join(txts),
                            (12, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (255,255,255), 1, cv2.LINE_AA)

            if out is not None:
                out.write(frame_for_write)

            t_enc1 = now()
            e2e_enc_ms = (t_enc1 - t_e2e0) * 1000.0
            if METRICS_ENABLE:
                PROF.upd_e2e_enc(e2e_enc_ms)
                trig_name = phase
                PROF.commit_frame(
                    frame_idx=int(src_frame_idx),
                    phase=phase,
                    cls_idx=cls_idx if cls_idx is not None else np.nan,
                    trigger_idx=trigger_idx if trigger_idx is not None else np.nan,
                    trigger_name=trig_name,
                    mode_changed=bool(mode_changed),
                    active_k_before=np.nan,
                    active_k_after=np.nan,
                    next_fire_idx_after=np.nan,
                    dropped_src_total=int(dropped_src),
                    e2e_infer_ms=float(e2e_infer_ms),
                    e2e_infer_display_ms=float(e2e_disp_ms),
                    e2e_infer_display_encode_ms=float(e2e_enc_ms),
                    dev_pred=float(dev_pred) if not np.isnan(dev_pred) else np.nan,
                    tau=float(tau),
                    fire_reason=fire_reason,
                    hold_count_since_fire=int(hold_age),
                )

            written += 1
            processed += 1
            pbar.update(int(dropped_now) + 1)
    finally:
        pbar.close()
        if out is not None:
            out.release()
        cap.release()

    t_total = now() - t0_pipeline
    fps_eff_total = written / t_total if t_total > 0 else 0.0

    print(f"=== RESUMEN (ADAPTIVE SCHEDULER | tau={ADAPTIVE_TAU:.4f} | medición global estilo benchmark | comparable={BENCHMARK_STYLE_NO_AUTOCAST}) ===")
    print(f"Fuente fps:             {fps_src:.2f}")
    print(f"Frames fuente aprox:    {frame_count_prop if frame_count_prop>0 else 'desconocido'}")
    print(f"Resolución entrada:     {w_src}x{h_src}")
    print(f"Resolución salida/disp: {TARGET_W}x{TARGET_H}")
    print(f"Modo sin autocast estilo benchmark: {BENCHMARK_STYLE_NO_AUTOCAST}")
    print(f"TF32 matmul activado: {getattr(torch.backends.cuda.matmul, 'allow_tf32', None)}")
    print(f"TF32 cuDNN activado: {getattr(torch.backends.cudnn, 'allow_tf32', None)}")
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"Autocast FP16:           {not BENCHMARK_STYLE_NO_AUTOCAST}")
    print(f"Tau scheduler:          {ADAPTIVE_TAU:.6f}")
    print(f"Max HOLD frames:        {MAX_HOLD_FRAMES}")
    print(f"Frames procesados:      {processed}")
    print(f"Frames descartados:     {dropped_src}")
    print(f"Último src_idx:         {last_src_idx}")
    print(f"FIRE:                   {fire_cnt}")
    print(f"HOLD:                   {hold_cnt}")
    print(f"Fire ratio:             {(fire_cnt / max(1, fire_cnt + hold_cnt)):.4f}")
    print(f"Errores inferencia:     {errors}")
    print(f"Escritos (total):       {written}")
    print(f"FPS efectivo (flujo):   {fps_eff_total:.2f}")

    summary_df = pd.DataFrame()
    details_df = pd.DataFrame()

    if METRICS_ENABLE:
        details_df = PROF.details_df()
        summary_df = PROF.summary_df(
            method_name=("adaptive_scheduler_realistic_benchmark_comparable" if BENCHMARK_STYLE_NO_AUTOCAST else "adaptive_scheduler_realistic_optimized"),
            input_video=input_path,
            output_video=output_path,
            fps_src=fps_src,
            w_src=w_src, h_src=h_src,
            target_w=TARGET_W, target_h=TARGET_H,
            processed=processed, written=written, dropped_src=dropped_src,
            fire_cnt=fire_cnt, hold_cnt=hold_cnt, errors=errors,
            last_src_idx=last_src_idx,
            k_straight=-1,
            k_curve=-1,
            final_trigger_name=f'TAU_{ADAPTIVE_TAU:.4f}',
            next_fire_idx_final=-1,
            config_path=CONFIG,
            ckpt_path=CKPT,
        )
        if len(summary_df) > 0:
            summary_df['tau'] = float(ADAPTIVE_TAU)
            summary_df['max_hold_frames'] = np.nan if MAX_HOLD_FRAMES is None else int(MAX_HOLD_FRAMES)

        print("--- FPS derivados (media global) ---")
        fgpu = PROF.fps_gpu(); fgpuA = PROF.fps_gpu_adj()
        fe2e = PROF.fps_e2e(); fdi = PROF.fps_e2e_disp(); fen = PROF.fps_e2e_enc()
        if fgpu is not None:  print(f"GPU-only FPS (global):                 {fgpu:.2f}")
        if fgpuA is not None: print(f"GPU-only FPS ajustado (global):       {fgpuA:.2f}")
        if fe2e is not None:  print(f"E2E FPS (infer, global):              {fe2e:.2f}")
        if fdi is not None:   print(f"E2E FPS (infer + display, global):    {fdi:.2f}")
        if fen is not None:   print(f"E2E FPS (infer + display + encode):   {fen:.2f}")
        print(f"[Calibración] event_overhead≈{PROF.calib_event_ms:.4f} ms, sync_stall_ref≈{PROF.calib_stall_ms:.4f} ms")
        if PROF.mean_gpu_stall() is not None:
            print(f"[Runtime]     stall_medio_global≈{PROF.mean_gpu_stall():.4f} ms")
        if len(summary_df) > 0:
            display(summary_df)

        if SAVE_RESULTS:
            os.makedirs(RESULTS_DIR, exist_ok=True)
            summary_df.to_csv(SUMMARY_CSV, index=False)
            details_df.to_csv(DETAILS_CSV, index=False)
            print('CSV resumen :', SUMMARY_CSV)
            print('CSV detalle :', DETAILS_CSV)
        if SAVE_VIDEO:
            print('Video guardado:', output_path)

    return {
        'summary_df': summary_df,
        'details_df': details_df,
        'fps_effective_flow': fps_eff_total,
        'processed': processed,
        'written': written,
        'dropped_src': dropped_src,
        'fire_cnt': fire_cnt,
        'hold_cnt': hold_cnt,
        'errors': errors,
    }


# ------------------------------------------------------------
# 9) Run
# ------------------------------------------------------------
def _resolve_ckpt_path(config_path, ckpt_path=None):
    if ckpt_path is not None and os.path.isfile(ckpt_path):
        return ckpt_path
    if ckpt_path is not None and ('*' in ckpt_path or '?' in ckpt_path):
        matches = sorted(glob.glob(ckpt_path))
        if matches:
            return matches[-1]
    cfg = Config.fromfile(config_path)
    work_dir = cfg.get('work_dir', None)
    if work_dir is None:
        raise RuntimeError('El config no define work_dir y CKPT=None.')
    patterns = ['latest.pth', 'best_cls_acc_cls_top1*.pth', 'best_seg_mIoU*.pth', 'best*.pth', 'iter_*.pth', '*.pth']
    candidates = []
    for pat in patterns:
        candidates.extend(glob.glob(os.path.join(work_dir, pat)))
    if not candidates:
        raise RuntimeError(f'No encontré checkpoints en work_dir={work_dir}')
    candidates = sorted(set(candidates))
    return candidates[-1]

def _disable_pretrained_init_for_test(cfg):
    """Avoids loading open-mmlab://resnet50_v1c during testing; checkpoint weights are used."""
    try:
        cfg.model.backbone.init_cfg = None
    except Exception:
        pass
    try:
        cfg.model.backbone.backbone_cfg.init_cfg = None
    except Exception:
        pass
    return cfg

def build_adaptive_scheduler_pipeline():
    register_all_modules()
    cfg = Config.fromfile(CONFIG)
    cfg = _disable_pretrained_init_for_test(cfg)

    # If the config defines tau, it can be used as the default.
    # Read the stateful configuration from the config, but DO NOT overwrite ADAPTIVE_TAU.
# In this notebook, tau is controlled by the top-level ADAPTIVE_TAU variable.
    try:
        global ADAPTIVE_TAU, MAX_HOLD_FRAMES, FIRST_FRAME_FIRE
        st = cfg.get('adaptive_stateful_cfg', {})
        MAX_HOLD_FRAMES = st.get('max_hold_frames', MAX_HOLD_FRAMES)
        FIRST_FRAME_FIRE = bool(st.get('first_frame_fire', FIRST_FRAME_FIRE))

        print(f"[TAU] tau en config = {st.get('tau', None)}")
        print(f"[TAU] tau usado por notebook = {ADAPTIVE_TAU}")

    except Exception as e:
        print(f"[TAU] No se pudo leer adaptive_stateful_cfg desde config: {e}")

    init_default_scope(cfg.get('default_scope', 'mmseg'))
    model = MODELS.build(cfg.model)

    resolved_ckpt = _resolve_ckpt_path(CONFIG, CKPT)
    load_checkpoint(model, resolved_ckpt, map_location='cpu')
    model.cfg = cfg
    model.to(DEVICE)
    model.eval()
    return dict(model=model, cfg=cfg, resolved_ckpt=resolved_ckpt)

if __name__ == '__main__':
    pipe = build_adaptive_scheduler_pipeline()
    model = pipe['model']
    print(f"Loads checkpoint by local backend from path: {pipe['resolved_ckpt']}")
    print(f"Tau adaptive scheduler: {ADAPTIVE_TAU:.6f}")
    print(f"Max HOLD frames: {MAX_HOLD_FRAMES}")
    print(f"Modo sin autocast estilo benchmark: {BENCHMARK_STYLE_NO_AUTOCAST}")
    print(f"TF32 matmul activado: {getattr(torch.backends.cuda.matmul, 'allow_tf32', None)}")
    print(f"TF32 cuDNN activado: {getattr(torch.backends.cudnn, 'allow_tf32', None)}")
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")

    infer = AdaptiveSchedulerStatefulInferencer(
        model,
        tau=ADAPTIVE_TAU,
        max_hold_frames=MAX_HOLD_FRAMES,
        first_frame_fire=FIRST_FRAME_FIRE
    )
    print("✅ Adaptive scheduler stateful instalado: FIRE/HOLD por dev_pred > tau.")

    warmup_full_loop(INPUT_VIDEO, max_frames=60, infer_call=infer)
    infer.reset_state()

    def video_main():
        return video_to_video_preserve_duration(
            INPUT_VIDEO,
            output_path=OUTPUT_VIDEO,
            enforce_deadline=True,
            write_unprocessed_on_error=True,
            annotate_osd=False,
            fps_override=10.0,
            infer_call=infer,
            ctx_clock=None
        )

    video_main()
